Install dependencies and packages


In [1]:
!pip install pandas crewai langchain huggingface_hub google-colab

In [2]:
!pip install --upgrade pip setuptools wheel
!pip install --upgrade crewai litellm langchain huggingface_hub


  Using cached litellm-1.74.7-py3-none-any.whl.metadata (40 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 84.1 MB/s eta 0:00:00
  Attempting uninstall: litellm
    Found existing installation: litellm 1.72.6
    Uninstalling litellm-1.72.6:
      Successfully uninstalled litellm-1.72.6
  Attempting uninstall: crewai
    Found existing installation: crewai 0.148.0
    Uninstalling crewai-0.148.0:
      Successfully uninstalled crewai-0.148.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [crewai]


In [3]:
!pip install --upgrade pip setuptools wheel

In [4]:
!pip install --upgrade crewai litellm langchain huggingface_hub

  Using cached litellm-1.74.7-py3-none-any.whl.metadata (40 kB)


In [5]:
!pip install langchain-community


In [6]:
!pip install --upgrade pip setuptools wheel
!pip install crewai langchain-huggingface pandas google-colab


In [7]:
!pip install --upgrade openai langchain crewai


In [ ]:
import os
from google.colab import userdata

# Access the secret and set environment variable for OpenAI
secret_value = userdata.get("OPENAI_API_KEY")
os.environ["OPENAI_API_KEY"] = secret_value

In [ ]:
!pip install --upgrade langchain openai


In [ ]:
import langchain
import openai
print(f"langchain version: {langchain.__version__}")
print(f"openai version: {openai.__version__}")


Market Research Agent Pipeline


In [8]:
# ── market_research_pipeline.py ──
import os

# 1) Override Litellm’s default model BEFORE anything else
os.environ["LITELLM_MODEL"] = "gpt-3.5-turbo"

# 2) Load OpenAI key
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# ── imports ──
import pandas as pd
from langchain.chat_models import ChatOpenAI
from langchain.schema import HumanMessage
from google.colab import files

# LLMTool wrapper
class LLMTool:
    def __init__(self, model_name: str):
        self.client = ChatOpenAI(model_name=model_name, temperature=0.0)
    def run(self, prompt: str) -> str:
        return self.client([HumanMessage(content=prompt)]).content

llm_tool = LLMTool(model_name=os.getenv("LITELLM_MODEL"))

# Agents
class DataLoaderAgent:
    def run(self, file_path: str) -> pd.DataFrame:
        df = pd.read_csv(file_path)
        return df

class DataAnalyzerAgent:
    def run(self, df: pd.DataFrame) -> str:
        preview = df.head().to_string()
        stats   = df.describe().to_string()
        prompt = (
            "You are a market research analyst.\n\n"
            f"Data preview:\n{preview}\n\n"
            f"Summary statistics:\n{stats}\n\n"
            "Give bullet‑point insights."
        )
        return llm_tool.run(prompt)

class ReportCreatorAgent:
    def run(self, insights: str) -> str:
        prompt = (
            "You are a BI writer.\n\n"
            f"Insights:\n{insights}\n\n"
            "Write an executive summary, key findings, and recommendations."
        )
        return llm_tool.run(prompt)

class PresentationCreatorAgent:
    def run(self, report: str) -> str:
        prompt = (
            "You are a presentation designer.\n\n"
            f"Report:\n{report}\n\n"
            "Outline a 10‑slide deck, with title and 3‑5 bullet points per slide."
        )
        return llm_tool.run(prompt)

# ── Orchestrate & print each result ──
if __name__ == "__main__":
    print("Upload your CSV:")
    uploaded = files.upload()
    csv_path = next(iter(uploaded.keys()))

    # 1) Load
    loader = DataLoaderAgent()
    df = loader.run(csv_path)
    print("\n--- DataLoaderAgent Output (first 5 rows) ---")
    print(df.head())

    # 2) Analyze
    analyzer = DataAnalyzerAgent()
    insights = analyzer.run(df)
    print("\n--- DataAnalyzerAgent Output (insights) ---")
    print(insights)

    # 3) Report
    reporter = ReportCreatorAgent()
    report = reporter.run(insights)
    print("\n--- ReportCreatorAgent Output (report) ---")
    print(report)

    # 4) Presentation outline
    presenter = PresentationCreatorAgent()
    outline = presenter.run(report)
    print("\n--- PresentationCreatorAgent Output (10‑slide outline) ---")
    print(outline)


/tmp/ipython-input-8-251909047.py:20: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import ChatOpenAI``.
  self.client = ChatOpenAI(model_name=model_name, temperature=0.0)


Upload your CSV:


Saving apple_smartwatch_survey_2000.csv to apple_smartwatch_survey_2000 (18).csv

--- DataLoaderAgent Output (first 5 rows) ---
   RespondentID AgeGroup             Gender UsageFrequency  \
0             1    35-44               Male        Monthly   
1             2    18-24             Female          Daily   
2             3    45-54  Prefer not to say          Daily   
3             4    25-34               Male        Monthly   
4             5    35-44         Non-binary          Never   

   SatisfactionRating                                   FavoriteFeatures  \
0                   6                   Fitness Tracking; Sleep Tracking   
1                   3                              Water Resistance; ECG   
2                   5                      Music Control; Sleep Tracking   
3                   4  Sleep Tracking; Water Resistance; Notification...   
4                   2                                                NaN   

  PurchaseIntent  LikelihoodToRecommend Pr

/tmp/ipython-input-8-251909047.py:22: LangChainDeprecationWarning: The method `BaseChatModel.__call__` was deprecated in langchain-core 0.1.7 and will be removed in 1.0. Use :meth:`~invoke` instead.
  return self.client([HumanMessage(content=prompt)]).content



--- DataAnalyzerAgent Output (insights) ---
- The data includes information on respondents' age group, gender, usage frequency, satisfaction rating, favorite features, purchase intent, likelihood to recommend, price sensitivity, preferred color, and comments.
- The average satisfaction rating is 6.26 out of 10, indicating a moderate level of satisfaction among respondents.
- The average likelihood to recommend is 5.26 out of 10, suggesting that respondents are somewhat likely to recommend the product.
- The most common favorite features mentioned are fitness tracking, sleep tracking, water resistance, ECG, music control, and notifications.
- The majority of respondents have a low to medium price sensitivity, indicating that they are willing to spend a moderate amount on a smartwatch.
- Black and gold are the preferred colors among respondents, with space gray also being mentioned.
- Comments provided by respondents highlight the sleek design, accuracy, water resistance, music control,

Created dummy data for survery


In [ ]:
import pandas as pd
import random

random.seed(42)

# Define possible values
age_groups = ['18-24', '25-34', '35-44', '45-54', '55+']
genders = ['Male', 'Female', 'Non-binary', 'Prefer not to say']
usage_freq = ['Daily', 'Weekly', 'Monthly', 'Rarely', 'Never']
features = [
    'Heart Rate Monitoring', 'Sleep Tracking', 'Fitness Tracking',
    'GPS', 'Notifications', 'Music Control', 'ECG',
    'Fall Detection', 'Water Resistance', 'Voice Assistant'
]
purchase_intent = ['Yes', 'No', 'Maybe']
price_sensitivity = ['Low', 'Medium', 'High']
preferred_colors = ['Black', 'Silver', 'Gold', 'Space Gray', 'Blue']
comments_pool = [
    "Love the sleek design and accuracy!",
    "Great for workouts but battery life could improve.",
    "Useful, but a bit pricey.",
    "Not interested, prefer traditional watches.",
    "Highly recommend, especially for health monitoring.",
    "Good features but sometimes laggy.",
    "Comfortable and reliable.",
    "Considering buying but waiting for discounts.",
    "Too expensive for my needs.",
    "Perfect companion for workouts and music.",
    "Wish it had longer battery life.",
    "The interface is intuitive and easy to use.",
    "Would like more watch face options.",
    "Helpful health alerts, very motivating.",
    "Sometimes connectivity issues with my phone."
]

data_rows = []

for i in range(1, 2001):
    age = random.choices(age_groups, weights=[0.2, 0.35, 0.25, 0.15, 0.05])[0]
    gender = random.choice(genders)
    usage = random.choices(usage_freq, weights=[0.4, 0.3, 0.15, 0.1, 0.05])[0]
    satisfaction = random.randint(3, 10) if usage != 'Never' else random.randint(1, 6)
    fav_feat_count = random.randint(1, 4) if usage != 'Never' else 0
    fav_feats = random.sample(features, fav_feat_count)
    fav_feats_str = "; ".join(fav_feats) if fav_feat_count > 0 else "N/A"
    intent = random.choices(purchase_intent, weights=[0.5, 0.2, 0.3])[0]
    recommend = random.randint(4, 10) if intent == 'Yes' else random.randint(1, 6)
    price_sens = random.choices(price_sensitivity, weights=[0.3, 0.5, 0.2])[0]
    color = random.choice(preferred_colors)
    comment = random.choice(comments_pool) if usage != 'Never' else "Not interested in smartwatch products."

    row = {
        "RespondentID": i,
        "AgeGroup": age,
        "Gender": gender,
        "UsageFrequency": usage,
        "SatisfactionRating": satisfaction,
        "FavoriteFeatures": fav_feats_str,
        "PurchaseIntent": intent,
        "LikelihoodToRecommend": recommend,
        "PriceSensitivity": price_sens,
        "PreferredColor": color,
        "Comments": comment
    }
    data_rows.append(row)

df = pd.DataFrame(data_rows)
import os

# Print current working directory
print("Current directory:", os.getcwd())

# List files in current directory
print("Files here:")
print(os.listdir())


# Save to CSV
df.to_csv("apple_smartwatch_survey_2000.csv", index=False)

print("Generated dataset saved as apple_smartwatch_survey_2000.csv")


In [ ]:
from google.colab import files

files.download("apple_smartwatch_survey_2000.csv")